In [ ]:
import numpy as np

from ase import io

from matplotlib import pyplot as plt
from pfcc_extras.visualize.view import view_ngl

In [ ]:
traj = io.Trajectory('HKUST-1_PFOA_relaxed.traj')

images = [atoms for atoms in traj]

MOF_mask = traj[0].get_tags() == 0
PFAS_mask = traj[0].get_tags() == 1

mask = np.outer(MOF_mask, PFAS_mask)

for atoms in images:
    distances = atoms.get_all_distances(mic=True)
    atoms.info['MOF-PFAS_distance'] = distances[mask].min()

In [ ]:
queries = {'Energy [eV]': lambda atoms: [atoms.get_total_energy(), atoms.get_potential_energy()],
           'Volume [Å3]': lambda atoms: atoms.get_volume(),
           'MOF PFAS distance [Å]': lambda atoms: atoms.info['MOF-PFAS_distance'],
          }

fig, axs = plt.subplots(nrows=len(queries), figsize=(2*6.4, len(queries)*4.8), sharex=True)

if isinstance(axs, plt.Axes):
    axs = [axs]

time = [0]

results = {query: [] for query in queries}

for i, atoms in enumerate(images):
    for query in queries:
        results[query].append(queries[query](atoms))

for i, query in enumerate(queries):
    axs[i].plot(results[query])
    axs[i].set_prop_cycle(None)

for i, query in enumerate(queries):
    axs[i].set_ylabel(query)

view_ngl(images)